# 01a · Il primo modello

In questo notebook costruiamo il primo NIDS basato su machine learning: dai flussi di rete a un classificatore che decide se un flusso è **benigno** o un **attacco**.

Lungo la strada incontreremo le scelte che si fanno prima ancora di addestrare un modello: quali colonne usare, come trattare i dati sporchi, come dividere i dati per valutare il risultato. Nessuna di queste scelte è neutra.

**Come usare il notebook.** Esegui le celle in ordine con `Shift + Invio`. I riquadri **Prova tu** sono piccole modifiche da fare in aula. Le **Sfide** in fondo sono da fare a casa.

## 0. Setup

In [ ]:
# Cella di setup: eseguila per prima, ogni volta che apri il notebook.
import subprocess, sys
from pathlib import Path

REPO = "salvatore-drago/nids-ml-lab"

if "google.colab" in sys.modules:
    root = Path("/content/nids-ml-lab")
    if not root.exists():
        url = f"https://github.com/{REPO}.git"
        try:  # repository privato: token nei Secrets di Colab, con nome GITHUB_TOKEN
            from google.colab import userdata
            url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/{REPO}.git"
        except Exception:
            pass
        if subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(root)]).returncode:
            raise RuntimeError("Clone del repository non riuscito: controlla la connessione o l'accesso.")
    else:
        subprocess.run(["git", "-C", str(root), "pull", "-q"])
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "nidslab").is_dir()), None)
    if root is None:
        raise RuntimeError("Apri il notebook dalla cartella notebooks/ del repository nids-ml-lab.")

if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import nidslab as nl

nl.ensure_packages()
nl.set_style()
print("Repository:", root)

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text

SEED = 42

## 1. Il dataset

Ogni riga è un **flusso**: l'insieme dei pacchetti scambiati tra due host con lo stesso indirizzo, porta e protocollo, riassunto in numeri. Le colonne sono di tre tipi:
- **identificativi**: chi parla con chi, e quando;
- **feature di traffico**: come si comporta la conversazione (durata, pacchetti, byte, flag TCP, tempi tra pacchetti...);
- **etichette**: cosa è successo davvero, e in che giorno.

In [ ]:
df = nl.load_static()
features = nl.feature_columns(df)

groups = {
    "Identificativi": nl.ID_COLS,
    "Categoriche": nl.CATEGORICAL_COLS,
    "Feature di traffico": features,
    "Etichette e metadati": nl.LABEL_COLS,
}
print(f"{len(df):,} flussi\n")
for name, cols in groups.items():
    preview = ", ".join(cols[:5]) + (", ..." if len(cols) > 5 else "")
    print(f"{name:<22} {len(cols):>3} colonne   {preview}")

Ogni giorno della cattura ha i suoi attacchi.

In [ ]:
pd.crosstab(df["family"], df["day"])

> **Da notare.** Ogni famiglia d'attacco compare in **un solo giorno**, e il lunedì contiene solo traffico benigno. Il mercoledì quasi un flusso su tre è un DoS. Ci tornerà utile nella seconda esercitazione: addestrare un modello sui primi giorni e usarlo sui successivi significa chiedergli di riconoscere attacchi che non ha mai visto.

## 2. Pulizia: ogni scelta ha un costo

Prima di addestrare un modello dobbiamo decidere quali colonne e quali righe usare. Per ogni scelta, chiediamoci anche: **questa operazione si potrebbe fare allo stesso modo sul traffico reale, quando il sistema è in funzione?**

### 2.1 Gli identificativi

Gli identificativi dicono *chi* sta parlando, non *come*. Guardate il `flow_id`: contiene indirizzi, porte, protocollo e perfino l'istante di inizio.

In [ ]:
df["flow_id"].iloc[0]

Un modello che usa queste colonne può imparare che "l'host X attacca" invece di imparare com'è fatto un attacco. Funzionerebbe sul dataset e fallirebbe appena l'attaccante cambia indirizzo. Per questo `nl.feature_columns()` le esclude, insieme a `protocol` (che non è numerica) e alle etichette.

### 2.2 Colonne costanti

Una colonna che vale sempre lo stesso numero non aiuta a distinguere niente.

In [ ]:
nunique = df[features].nunique()
constant = nunique[nunique == 1].index.tolist()
print(f"{len(constant)} colonne costanti: {constant}")

features = [c for c in features if c not in constant]
print(f"Restano {len(features)} feature")

> **Da notare.** Le colonne costanti sono solo quelle del flag TCP **URG**: è un meccanismo previsto dal protocollo che oggi quasi nessuna applicazione usa, e in tutta la cattura non compare mai. Una colonna costante non è sempre un errore: a volte è una caratteristica del protocollo che in quella rete non si presenta.

### 2.3 Valori mancanti e anomali

In [ ]:
X_all = df[features]
print("Valori mancanti:", int(X_all.isna().sum().sum()))
print("Valori infiniti:", int(np.isinf(X_all.to_numpy(dtype=float)).sum()))

negative = (X_all < 0).sum()
negative = negative[negative > 0]
print(f"\nColonne con valori negativi: {len(negative)}")
print(negative.to_string())

Un tempo di attività negativo, o un intervallo negativo tra due pacchetti consecutivi, non ha senso fisico: sono **artefatti dello strumento** che ha estratto i flussi dai pacchetti. Riguardano una piccola quota dei flussi, e gli alberi di decisione non ne soffrono, perché ragionano per soglie e non per distanze. Per ora li lasciamo, ma è il tipo di dettaglio che va sempre controllato.

> **Da notare.** I valori negativi di `active_min`, `active_max` e `active_mean` riguardano 7.071 flussi, circa il 2,4% del totale. Gli intervalli minimi negativi (`packet_IAT_min`, `bwd_packets_IAT_min`) indicano pacchetti registrati con un timestamp precedente a quello del pacchetto che li precede.

### 2.4 Duplicati

Due flussi con **tutte** le feature identiche sono, per il modello, lo stesso esempio. Contiamoli, e contiamo anche un caso più subdolo: flussi con feature identiche ma **etichette diverse**.

In [ ]:
dup_mask = df.duplicated(subset=features + ["label"])
print(f"Righe duplicate (feature ed etichetta identiche): {dup_mask.sum():,} ({dup_mask.mean():.1%})")

# Raggruppiamo le righe con feature identiche usando un'impronta (hash) di ogni riga.
fingerprint = pd.util.hash_pandas_object(df[features], index=False)
labels_per_fingerprint = df.groupby(fingerprint)["label"].transform("nunique")
conflict = labels_per_fingerprint > 1
print(f"Righe con feature identiche ma etichette diverse: {conflict.sum():,}")

pd.DataFrame({
    "duplicati": df.loc[dup_mask, "label"].value_counts(),
    "in_conflitto": df.loc[conflict, "label"].value_counts(),
    "totale": df["label"].value_counts(),
}).query("duplicati > 0 or in_conflitto > 0")

> **Da notare.**
> - Un flusso su cinque è un duplicato.
> - **Port Scan è duplicato al 90%**: uno scan manda lo stesso pacchetto a porte diverse e riceve sempre la stessa risposta. Tolte le porte, che sono identificativi, i flussi diventano indistinguibili.
> - **Un terzo dei flussi DoS Hulk è in conflitto** (8.099 su 24.447), e dall'altra parte ci sono 878 flussi benigni in conflitto: migliaia di flussi Hulk sono identici, feature per feature, a flussi benigni. Nella prossima sezione scopriremo perché.

I duplicati esatti li togliamo: altrimenti la stessa riga può finire sia nei dati di addestramento sia in quelli di test, e il modello verrebbe "interrogato" su esempi che ha già visto.

Le righe in conflitto sono più interessanti: **nessun modello può classificarle tutte correttamente**, perché lo stesso input ha due risposte diverse. È rumore nelle etichette, e mette un tetto alle prestazioni possibili.

Perché allora non le togliamo? Per riconoscere una riga in conflitto serve la sua **etichetta**, e quando il sistema è in funzione le etichette non ci sono: arriva un flusso, e nessuno sa se esiste un flusso identico con un'etichetta diversa. Toglierle dai dati di test significherebbe togliere proprio i casi difficili che il sistema reale incontrerà, e i risultati sembrerebbero migliori di quanto sono. Dai dati di addestramento invece si possono togliere: è la Sfida 4.

### 2.5 I flussi di un solo pacchetto

Una conversazione di un pacchetto solo è strana. Quanti sono, classe per classe?

In [ ]:
single = df["packets_count"] == 1
single_by_class = pd.DataFrame({
    "flussi": df["label"].value_counts(),
    "da_1_pacchetto": df.loc[single, "label"].value_counts(),
})
single_by_class["quota"] = (single_by_class["da_1_pacchetto"] / single_by_class["flussi"]).round(3)
single_by_class.sort_values("quota", ascending=False)

> **Da notare.** Circa **metà dei web attack e di SSH-Patator** sono flussi di un pacchetto, e sono anche le classi più piccole. DDoS e Port Scan invece non ne hanno quasi. Heartbleed ne ha 4 su 12.

E che pacchetti sono? Guardiamo i flag TCP dei flussi di un solo pacchetto.

In [ ]:
flag_cols = ["syn_flag_counts", "fin_flag_counts", "rst_flag_counts", "ack_flag_counts", "psh_flag_counts"]
tcp_single = df.loc[single & (df["protocol"] == "TCP"), flag_cols] > 0
flag_names = tcp_single.apply(
    lambda r: "+".join(c.split("_")[0].upper() for c in flag_cols if r[c]) or "nessun flag", axis=1)
flag_names.value_counts()

Quasi nessuno ha il flag **SYN**, quello che apre una connessione: sono **pezzi della chiusura** di connessioni già iniziate.

Il motivo sta nello strumento che ha estratto i flussi, che chiude un flusso TCP al primo pacchetto FIN o RST. Una connessione TCP però si chiude con uno scambio: FIN da una parte, FIN dall'altra, e un **ACK finale**. I pacchetti che arrivano dopo il primo FIN, oppure dopo un RST, aprono quindi un flusso nuovo, che resta di un pacchetto solo. Lo stesso difetto era stato documentato da Engelen, Rimmer e Joosen (2021) per lo strumento usato nella versione originale di questo dataset.

> **Da notare.**
> - Più della metà dei frammenti è un **ACK isolato**: l'ultimo pacchetto della chiusura TCP. Quasi tutti gli altri contengono un RST, cioè una connessione interrotta.
> - Solo uno su mille porta il **SYN**. Un SYN senza seguito è un tentativo di apertura rimasto senza risposta: una porta filtrata da un firewall, un host spento, oppure un port scan che manda un solo tentativo per porta. Un client normale, se non riceve risposta, ritrasmette il SYN dopo circa un secondo, e i tentativi finiscono nello stesso flusso.
> - Se un firewall cambia regole durante la cattura, il traffico verso una porta passa da flussi completi a tentativi senza risposta da un giorno all'altro. È un cambiamento dell'ambiente, non degli utenti o degli attaccanti: ne riparleremo nella seconda esercitazione.

Ricordate le righe in conflitto della sezione 2.4? Quante di queste hanno un solo pacchetto?

In [ ]:
print(f"Righe in conflitto con un solo pacchetto:     {single[conflict].mean():.1%}")
print(f"Righe non in conflitto con un solo pacchetto: {single[~conflict].mean():.1%}")

> **Da notare.** **Tutte** le righe in conflitto sono flussi di un pacchetto. Un ACK isolato è identico sia che chiuda una connessione benigna sia che chiuda un attacco: a distinguerli resta solo l'etichetta.

Mettendo insieme i pezzi:
1. lo strumento di estrazione chiude il flusso troppo presto, e i pacchetti di chiusura diventano flussi di un pacchetto;
2. un frammento di chiusura è identico, feature per feature, in una connessione benigna e in una d'attacco;
3. in questo dataset l'etichetta d'attacco è assegnata in base all'**indirizzo IP dell'attaccante**, quindi frammenti identici ricevono etichette diverse;
4. nessun modello può classificarli tutti correttamente.

Le due strade possibili hanno entrambe un costo:
- **toglierli** è una regola applicabile anche quando il sistema è in funzione (guarda solo il numero di pacchetti, non l'etichetta) ed elimina tutti i conflitti. Però riduce proprio le classi già rare, e un sistema reale che li ignora non vedrebbe un attacco fatto solo di frammenti;
- **tenerli** significa addestrare il modello anche su frammenti che dell'attacco non hanno niente.

### 2.6 Le nostre decisioni

> **Prova tu.** Metti `REMOVE_SINGLE_PACKET = True` ed esegui di nuovo da questa cella in giù. Quali classi perdono più flussi? Cambiano i risultati del modello nella sezione 4?

In [ ]:
REMOVE_SINGLE_PACKET = False

keep = ~dup_mask
if REMOVE_SINGLE_PACKET:
    keep &= ~single
data = df[keep].reset_index(drop=True)

lost = 1 - data["label"].value_counts() / df["label"].value_counts()
print(f"Righe tenute: {len(data):,} su {len(df):,}")
print("\nQuota persa per classe:")
print(lost.sort_values(ascending=False).map("{:.1%}".format).to_string())

> **Da notare** (con `REMOVE_SINGLE_PACKET = False`). Togliendo solo i duplicati restano 236.320 flussi su 295.732. Perdono di più Port Scan (90%), DoS Hulk (44%) e SSH-Patator (40%).
>
> **Dopo il Prova tu** (con `True`). Togliere anche i frammenti pesa soprattutto sulle classi piccole: Web Brute Force passa dal 36% al 52% di flussi persi, Heartbleed da 2 a 4 flussi su 12, tutti i web attack superano il 40%. Su DoS Hulk invece non cambia quasi niente (dal 43,8% al 43,9%): i suoi frammenti erano quasi tutti già duplicati, lo stesso ACK isolato ripetuto migliaia di volte.

## 3. Come impara una macchina

Prima del dataset completo, un esempio che si può disegnare: solo flussi benigni e attacchi **DoS Hulk**, e solo **due feature**. Per scegliere le due feature lasciamo decidere a un piccolo albero quali usa di più.

I valori di traffico vanno da zero a milioni, quindi li mostriamo in scala logaritmica, cioè trasformati in $\log(1 + x)$. Sugli assi però sono scritti i valori originali.

In [ ]:
toy = data[data["label"].isin(["Benign", "DoS_Hulk"])]
n_per_class = min(1500, toy["label"].value_counts().loc[["Benign", "DoS_Hulk"]].min())
toy = toy.groupby("label", observed=True).sample(n_per_class, random_state=SEED)

probe = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(toy[features], toy["label_binary"])
top2 = pd.Series(probe.feature_importances_, index=features).nlargest(2).index.tolist()
print("Le due feature scelte:", top2)

X_toy = np.log1p(toy[top2].clip(lower=0)).to_numpy()
y_toy = toy["label_binary"].to_numpy()
Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    X_toy, y_toy, test_size=0.3, stratify=y_toy, random_state=SEED)

> **Da notare.** Tra oltre cento feature, l'albero sceglie **la dimensione degli header**: la deviazione standard di quelli di ritorno e la media su tutto il flusso. Cosa c'entra la dimensione di un header con un attacco DoS? Teniamo la domanda aperta e guardiamo il grafico.

L'albero divide il piano con tagli orizzontali e verticali: ogni taglio è una domanda del tipo "questa feature supera la soglia?". La linea grigia è la **frontiera di decisione**.

> **Prova tu.** Cambia `MAX_DEPTH` (1, 3, 8, `None`) e riesegui. Come cambia la frontiera? Confronta l'accuratezza sui dati di addestramento con quella sui dati di test, e guarda quante foglie servono.

In [ ]:
MAX_DEPTH = 2

tree = DecisionTreeClassifier(max_depth=MAX_DEPTH, random_state=SEED).fit(Xt_train, yt_train)
print(f"Accuratezza su addestramento: {tree.score(Xt_train, yt_train):.3f}")
print(f"Accuratezza su test:          {tree.score(Xt_test, yt_test):.3f}")
print(f"Profondità raggiunta:         {tree.get_depth()}")
print(f"Foglie dell'albero:           {tree.get_n_leaves()}")

ax = nl.plot_decision_regions(
    tree, Xt_test, yt_test, class_names=("Benign", "DoS Hulk"),
    feature_names=[f"{c} (scala log)" for c in top2],
    title=f"Albero di profondità {MAX_DEPTH}: dati di test")
nl.set_log1p_ticks(ax)
plt.show()

> **Da notare** (con `MAX_DEPTH = 2`).
> - I punti si dispongono su **due fasce orizzontali**: header medi di circa 32 byte e di circa 20 byte. 20 byte è un header TCP senza opzioni, 32 è lo stesso header con 12 byte di opzioni, tipicamente il *timestamp* TCP. Linux aggiunge quell'opzione di default, Windows no.
> - **Quasi tutti i DoS Hulk stanno nella fascia dei 32 byte**, mentre quella dei 20 byte contiene quasi solo traffico benigno. In questo dataset gli attacchi DoS partono da una macchina Linux, e gran parte del traffico benigno viene da client Windows.
> - È probabile quindi che il modello separi gli attacchi riconoscendo in buona parte **il sistema operativo della macchina che li lancia**: un'impronta di *chi* parla, rientrata dalla finestra dopo che avevamo tolto gli identificativi. Se l'attaccante usasse un'altra macchina, questa regola non funzionerebbe più.
> - La colonna a sinistra raccoglie i flussi in cui gli header di ritorno non variano mai, compresi i frammenti di un pacchetto: lì benigni e Hulk si sovrappongono, sono i conflitti della sezione 2.4.
> - Con due sole domande l'albero arriva già al 96% sul test.
>
> **Dopo il Prova tu.**
>
> | `MAX_DEPTH` | Addestramento | Test | Profondità | Foglie |
> |---|---|---|---|---|
> | 1 | 0,894 | 0,912 | 1 | 2 |
> | 2 | 0,955 | 0,963 | 2 | 4 |
> | 3 | 0,966 | 0,970 | 3 | 7 |
> | 8 | 0,976 | 0,971 | 8 | 34 |
> | `None` | 0,980 | 0,974 | 13 | 49 |
>
> - Da 7 a 49 foglie, l'accuratezza sul test guadagna **quattro millesimi**. La complessità cresce molto più dell'accuratezza.
> - Con profondità 8 e `None` la frontiera diventa una serie di **fasce sottilissime attorno a valori esatti** di dimensione dell'header. Non è una regola su com'è fatto un attacco: è un elenco di casi già visti. Un flusso con un header di due byte diverso finirebbe dalla parte sbagliata.
> - **Nemmeno senza limiti l'addestramento arriva a 1**: si ferma a 0,980 per i conflitti, punti identici con etichette diverse che nessun albero può separare.
> - Con profondità 1 e 2 il test supera l'addestramento: è solo variabilità, perché il test ha circa 900 punti.

Con poca profondità la frontiera è semplice e commette qualche errore. Con molta profondità diventa sempre più complicata per guadagnare pochissimo: l'albero sta **memorizzando** i dati di addestramento invece di imparare una regola. Questo è l'**overfitting**.

Perché allora l'accuratezza sul test non scende? Perché i dati di test vengono **dallo stesso dataset**: stesse macchine, stessi sistemi operativi, stessi valori di header. Il test misura quanto il modello generalizza *dentro questi dati*, non quanto generalizzerebbe su traffico nuovo. È una differenza importante, e ci torneremo alla fine del notebook.

## 4. Il primo modello

Torniamo al dataset completo, con tutte le feature. Il compito è **binario**: benigno (0) o attacco (1).

Usiamo solo il **20%** dei flussi per l'addestramento e il restante 80% per il test. La divisione è *stratificata* sull'etichetta di dettaglio, così ogni tipo di attacco, anche il più raro, compare in entrambe le parti.

In [ ]:
X = data[features]
y = data["label_binary"]

X_train, X_test, y_train, y_test, lab_train, lab_test = train_test_split(
    X, y, data["label"], train_size=0.2, stratify=data["label"], random_state=SEED)
print(f"Addestramento: {len(X_train):,} flussi   Test: {len(X_test):,} flussi")

model = DecisionTreeClassifier(random_state=SEED)
t0 = time.perf_counter()
model.fit(X_train, y_train)
print(f"Addestramento in {time.perf_counter() - t0:.1f} s, {model.get_n_leaves():,} foglie")

y_pred = model.predict(X_test)
print(f"\nAccuratezza: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=["Benign", "Attacco"], digits=4))

> **Da notare.**
> - Un albero addestrato su meno di 50 mila flussi, in pochi secondi, arriva al **99,6% di accuratezza**.
> - Nel test gli attacchi sono il 10,2% dei flussi, contro il 15,4% del campione completo: la rimozione dei duplicati ha tolto soprattutto port scan e DoS Hulk, e lo sbilanciamento è aumentato.

In [ ]:
names = np.array(["Benign", "Attacco"])
nl.plot_confusion_matrix(names[y_test], names[y_pred], labels=names,
                         title="Albero di decisione, task binario")
plt.show()

> **Da notare.** 345 falsi allarmi sono lo 0,2% del traffico benigno. Sembra pochissimo, ma su un giorno di traffico di una rete reale, con milioni di flussi, sarebbero migliaia di allarmi da controllare a mano.

### 4.1 Le prime spiegazioni del modello: le regole

Un albero si può leggere: è un elenco di regole "se... allora...". Per ogni flusso possiamo seguire il percorso dalla radice alla foglia e dire perché è stato classificato così. È la forma più semplice di **spiegazione** di un modello, e la ritroveremo nella seconda esercitazione, quando parleremo di modelli che non si possono leggere.

Quello appena addestrato ha centinaia di foglie, quindi per leggerlo ne addestriamo uno con soli tre livelli.

In [ ]:
short_tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train, y_train)
print(f"Accuratezza dell'albero a 3 livelli sul test: {short_tree.score(X_test, y_test):.4f}\n")
print(export_text(short_tree, feature_names=features, class_names=["Benign", "Attacco"], decimals=2))

> **Da notare.**
> - Con **tre domande** l'albero arriva al 97,4%, contro il 99,6% dell'albero completo con centinaia di foglie: le prime domande fanno quasi tutto il lavoro.
> - Gli **header** tornano anche qui (`min_header_bytes`, `std_header_bytes`): l'impronta del sistema operativo pesa anche nel modello completo.
> - Alcune regole si possono dire a parole: "il server ha risposto con un RST?" (`bwd_rst_flag_counts`), "le risposte del server hanno dimensioni molto variabili?" (`bwd_payload_bytes_variance`).
> - La soglia `bwd_packets_IAT_min <= 0` separa soprattutto i flussi in cui il server ha mandato meno di due pacchetti: in quel caso l'intervallo tra pacchetti di ritorno non esiste, e lo strumento di estrazione scrive zero.

### 4.2 Le prime spiegazioni del modello: l'importanza delle feature

L'importanza di una feature misura quanto le domande che la riguardano aiutano a separare le classi, sommato su tutto l'albero. Se le regole spiegano una singola decisione, l'importanza descrive il modello nel suo insieme.

In [ ]:
importance = pd.Series(model.feature_importances_, index=features).nlargest(10).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(importance.index, importance.values, color=nl.BENIGN_COLOR, height=0.6)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Importanza")
ax.set_title("Le 10 feature più importanti", loc="left")
plt.show()

> **Da notare.**
> - **Una sola feature prende il 72%** dell'importanza: è la domanda alla radice dell'albero, che separa per prima la maggior parte dei flussi e accumula quasi tutto il merito. Non significa che le altre siano inutili.
> - Questo modo di misurare l'importanza ha limiti noti: favorisce le feature con molti valori diversi, si calcola sui dati di addestramento, e quando due feature sono correlate il merito si divide tra loro in modo arbitrario.
> - Importanza non vuol dire causalità: dice su cosa il modello si basa, non perché un flusso sia un attacco.

### 4.3 Quali attacchi riconosce?

Il modello è binario, ma conosciamo il tipo di ogni attacco nel test. Per ogni tipo calcoliamo la **detection rate**, cioè la quota di flussi d'attacco classificati come attacco. Per il traffico benigno calcoliamo invece la quota di **falsi allarmi**.

In [ ]:
results = pd.DataFrame({"label": lab_test.to_numpy(), "pred": y_pred})
by_class = results.groupby("label", observed=True)["pred"].agg(flussi_test="size", classificati_attacco="mean")
by_class["classificati_attacco"] = by_class["classificati_attacco"].round(4)
by_class.sort_values("flussi_test", ascending=False)

> **Da notare.**
> - Gli attacchi più numerosi sono riconosciuti quasi tutti: DoS Hulk 99,3%, DDoS 99,7%, Port Scan 97,7%.
> - Quelli rari molto meno: Web XSS 81%, Web Brute Force 78%, Slowhttptest 77%, SQL Injection 8 su 15, **Heartbleed 2 su 8**. Con il 20% dei dati per l'addestramento, Heartbleed aveva 2 o 3 esempi da cui imparare.
> - Su 8 flussi, uno in più o in meno sposta la percentuale di 12,5 punti: per le classi rarissime queste cifre vanno lette come "bene" o "male", non al decimale.
> - L'accuratezza complessiva è dominata dal traffico benigno e dai tre attacchi più grandi. Un modello al 99,6% può mancare **tre Heartbleed su quattro e metà delle SQL injection**.

## 5. Troppo bello per essere vero?

Prima di festeggiare, un esperimento. Addestriamo lo stesso albero su porzioni sempre più piccole dei dati e lo valutiamo **sempre sullo stesso test**.

In [ ]:
rows = []
for share in [0.01, 0.05, 0.20]:
    if share < 0.20:
        X_sub, _, y_sub, _ = train_test_split(
            X_train, y_train, train_size=share / 0.20, stratify=y_train, random_state=SEED)
    else:
        X_sub, y_sub = X_train, y_train
    m = DecisionTreeClassifier(random_state=SEED).fit(X_sub, y_sub)
    p = m.predict(X_test)
    rows.append({
        "quota dei dati per l'addestramento": f"{share:.0%}",
        "flussi di addestramento": len(X_sub),
        "accuratezza": round(accuracy_score(y_test, p), 4),
        "attacchi riconosciuti": round(p[y_test == 1].mean(), 4),
        "falsi allarmi": round(p[y_test == 0].mean(), 4),
    })
pd.DataFrame(rows)

> **Da notare.**
> - Con l'**1% dei dati**, poco più di 2 mila flussi, l'albero arriva già al **98,8%** di accuratezza. Moltiplicare i dati per venti fa guadagnare meno di un punto: il compito, così come l'abbiamo impostato, è sorprendentemente facile.
> - Nello stesso passaggio, però, i **falsi allarmi scendono quasi di quattro volte** (dallo 0,76% allo 0,20%) e gli attacchi persi passano da circa 1 su 20 a meno di 1 su 50. Su una rete reale è la differenza tra un sistema utilizzabile e uno che gli operatori smettono di guardare. L'accuratezza quasi non se ne accorge.

Prima di fidarvi di questi numeri, provate a rispondere:

1. **Come abbiamo diviso addestramento e test?** A caso, mescolando flussi di tutti i giorni. Un attacco dura minuti e genera migliaia di flussi quasi identici: se alcuni finiscono nell'addestramento e altri nel test, il modello è interrogato su un attacco che ha già visto, anche se non su quella riga precisa. Succederebbe così anche con un attacco nuovo, mai visto prima? E con un attaccante che usa un altro sistema operativo?
2. **Abbiamo provato una volta sola.** Con un altro `SEED`, cioè un'altra divisione casuale, i risultati sarebbero gli stessi? Di quanto cambiano?
3. **L'accuratezza è il numero giusto da guardare?** Che accuratezza otterrebbe un modello che risponde sempre "benigno"?

Torneremo sulla domanda 2 nel prossimo notebook, sulla 3 in `01c`, e sulla 1 nella seconda esercitazione.

---

## Sfide (a casa)

Ogni sfida ha una cella con dei `TODO` da completare e un controllo (`assert`) che ti dice se hai impostato le cose nel modo giusto. Esegui le sfide con `REMOVE_SINGLE_PACKET = False`, salvo dove indicato.

### Sfida 1: il protocollo

`protocol` (TCP o UDP) non è numerica, quindi l'abbiamo esclusa. Aggiungila al modello come due colonne 0/1 (*one-hot encoding*, con `pd.get_dummies`) e riaddestra l'albero sulla stessa divisione. Cambia qualcosa? La feature compare tra le più importanti?

In [ ]:
# TODO: costruisci X_proto unendo a X le colonne one-hot di data["protocol"]
X_proto = ...

assert {"protocol_TCP", "protocol_UDP"} <= set(X_proto.columns), "Mancano le colonne one-hot del protocollo"
assert len(X_proto) == len(X), "X_proto deve avere le stesse righe di X"

# TODO: addestra l'albero su X_proto.loc[X_train.index] e valutalo su X_proto.loc[X_test.index]

### Sfida 2: solo le porte

Addestra un albero usando **solo** `src_port` e `dst_port` (sono identificativi, quindi le avevamo escluse), con la stessa divisione di addestramento e test. Quanto arriva in alto? Perché un risultato alto qui sarebbe un problema e non una buona notizia?

In [ ]:
port_cols = ["src_port", "dst_port"]

# TODO: addestra un albero su data.loc[X_train.index, port_cols] e calcola l'accuratezza sul test
acc_ports = ...

assert 0 <= acc_ports <= 1, "acc_ports deve essere un'accuratezza tra 0 e 1"
print(f"Accuratezza usando solo le porte: {acc_ports:.4f}")

### Sfida 3: senza i flussi di un pacchetto

Ripeti le sezioni 2.6 e 4 con `REMOVE_SINGLE_PACKET = True` e confronta la tabella della sezione 4.3 con quella ottenuta tenendo i flussi di un pacchetto. Per quali attacchi cambia di più la detection rate? Scrivi in due righe quale delle due scelte faresti per un sistema reale, e perché.

In [ ]:
# TODO: salva le due tabelle della sezione 4.3 (con e senza flussi di un pacchetto)
by_class_with = ...
by_class_without = ...

assert "classificati_attacco" in by_class_with.columns and "classificati_attacco" in by_class_without.columns

comparison = pd.DataFrame({
    "con_1_pacchetto": by_class_with["classificati_attacco"],
    "senza_1_pacchetto": by_class_without["classificati_attacco"],
})
comparison["differenza"] = comparison["senza_1_pacchetto"] - comparison["con_1_pacchetto"]
comparison.sort_values("differenza")

### Sfida 4: togliere i conflitti solo dall'addestramento

Nella sezione 2.4 abbiamo visto che le righe in conflitto si possono togliere dai dati di addestramento, ma non da quelli di test. Fallo: individua le righe di `data` in conflitto (stesse feature, etichette diverse, come nella sezione 2.4), toglile **solo** dall'addestramento, riaddestra l'albero e valutalo sullo **stesso** `X_test` della sezione 4.

Confronta il risultato con quello della sezione 4. Migliorano i falsi allarmi? E la detection rate dei DoS Hulk, la classe con più conflitti?

In [ ]:
# TODO: per ogni riga di `data`, True se è in conflitto (come nella sezione 2.4, ma calcolato su `data`)
conflict_data = ...

assert len(conflict_data) == len(data), "conflict_data deve avere una riga per ogni riga di data"

# TODO: gli indici di X_train che NON sono in conflitto
train_clean_idx = ...

assert set(train_clean_idx) <= set(X_train.index), "Gli indici devono venire da X_train"
assert len(train_clean_idx) < len(X_train), "Dovresti aver tolto almeno una riga"

# TODO: addestra l'albero su X.loc[train_clean_idx] e valutalo su X_test